# <img src="https://raw.githubusercontent.com/iterative/datachain/main/docs/assets/datachain.svg" style="height: 32px; width: 32px; vertical-align: bottom"/> Tutorial: Working with Video Datasets

This tutorial explores techniques for managing video datasets.

**📋 Topics covered:**
1. Building a Video DataChain for the [Kinetics-700-2020](https://github.com/cvdfoundation/kinetics-dataset) dataset
2. Adding video file metadata
3. Extracting Video Fragments
4. Extracting Video Frames

## Setup

To start, install the dependencies:

In [1]:
%pip install -q 'datachain[video]'


[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Import all required modules:

In [2]:
from collections.abc import Iterator

from datachain import DataChain, C, Video, VideoFile, VideoFragment, VideoFrame
from datachain.lib.video import video_info

## Building a Video DataChain from a bucket

We will use the [Kinetics-700-2020](https://github.com/cvdfoundation/kinetics-dataset) dataset from Google Cloud Storage:

In [3]:
DATA_PATH = "gs://datachain-kinetics-700-2020/val/"

video_dc = (
    DataChain.from_storage(DATA_PATH, type="video", anon=True)
    .settings(parallel=4, cache=True)
    .filter(C("file.path").glob("*.mp4"))
    .save("kinetics-700-2020-val")
)

In [4]:
video_dc.count()

33966

In [5]:
video_dc.show(10)

,file,file,file,file,file,file,file,file
,source,path,size,version,etag,is_latest,last_modified,location
0,gs://datachain-kinetics-700-2020,val/abseiling/3E7Jib8Yq5M_000118_000128.mp4,3589116,1726417868815045,CMX1t96vxYgDEAE=,1,2024-09-15 16:31:08.818000+00:00,None
1,gs://datachain-kinetics-700-2020,val/abseiling/CPMhjRqrLxI_001552_001562.mp4,2592594,1726417867745860,CMTU9t2vxYgDEAE=,1,2024-09-15 16:31:07.748000+00:00,None
2,gs://datachain-kinetics-700-2020,val/abseiling/J_faWIyCzdc_000033_000043.mp4,2988831,1726417867716076,COzr9N2vxYgDEAE=,1,2024-09-15 16:31:07.718000+00:00,None
3,gs://datachain-kinetics-700-2020,val/abseiling/Kbmv94BqZzU_000072_000082.mp4,3315420,1726417867526305,CKGh6d2vxYgDEAE=,1,2024-09-15 16:31:07.528000+00:00,None
4,gs://datachain-kinetics-700-2020,val/abseiling/LFMlCcsLdKA_000126_000136.mp4,611088,1726417866815279,CK/uvd2vxYgDEAE=,1,2024-09-15 16:31:06.818000+00:00,None
5,gs://datachain-kinetics-700-2020,val/abseiling/MpnTZgY650k_000037_000047.mp4,493020,1726417866998681,CJmHyd2vxYgDEAE=,1,2024-09-15 16:31:07+00:00,None
6,gs://datachain-kinetics-700-2020,val/abseiling/ND-5jaduGFg_000024_000034.mp4,312869,1726417866659017,CMmptN2vxYgDEAE=,1,2024-09-15 16:31:06.661000+00:00,None
7,gs://datachain-kinetics-700-2020,val/abseiling/NHXo1PE12BU_000075_000085.mp4,512350,1726417867016976,CJCWyt2vxYgDEAE=,1,2024-09-15 16:31:07.019000+00:00,None
8,gs://datachain-kinetics-700-2020,val/abseiling/NkLeMkOcftM_000037_000047.mp4,228082,1726417866405677,CK3upN2vxYgDEAE=,1,2024-09-15 16:31:06.408000+00:00,None



[Limited by 10 rows]


### (OPTIONAL) Create a DataChain from a previously saved dataset

In [6]:
%%script false --no-raise-error  # skip this step

video_dc = DataChain.from_dataset("kinetics-700-2020-val")

print(video_dc.count())
video_dc.show(10)

### In this example, we will limit the video count to 10

This is enough to cover all topics while speeding up execution.

In [7]:
video_dc = video_dc.limit(10)

## Adding video file metadata

This code runs the built-in `video_info` mapper, which creates a new `meta` signal containing information about each video file:

In [8]:
video_meta_dc = video_dc.map(meta=video_info).save()

Preparing: 0 rows [00:00, ? rows/s]

Cleanup:   0%|          | 0/2 [00:00<?, ? tables/s]

Let's see signals schems in new DataChain:

In [9]:
video_meta_dc.print_schema()

 file: VideoFile@v1
     source: str
     path: str
     size: int
     version: str
     etag: str
     is_latest: bool
     last_modified: datetime
     location: Union[dict, list[dict], NoneType]
 meta: Video@v1
     width: int
     height: int
     fps: float
     duration: float
     frames: int
     format: str
     codec: str


In [10]:
video_meta_dc.show(10)

,file,file,file,file,file,file,file,file,meta,meta,meta,meta,meta,meta,meta
,source,path,size,version,etag,is_latest,last_modified,location,width,height,fps,duration,frames,format,codec
0,gs://datachain-kinetics-700-2020,val/abseiling/3E7Jib8Yq5M_000118_000128.mp4,3589116,1726417868815045,CMX1t96vxYgDEAE=,1,2024-09-15 16:31:08.818000+00:00,None,1280,720,25.000000,10.010793,250,"mov,mp4,m4a,3gp,3g2,mj2",h264
1,gs://datachain-kinetics-700-2020,val/abseiling/CPMhjRqrLxI_001552_001562.mp4,2592594,1726417867745860,CMTU9t2vxYgDEAE=,1,2024-09-15 16:31:07.748000+00:00,None,1280,720,29.970030,10.010000,300,"mov,mp4,m4a,3gp,3g2,mj2",h264
2,gs://datachain-kinetics-700-2020,val/abseiling/J_faWIyCzdc_000033_000043.mp4,2988831,1726417867716076,COzr9N2vxYgDEAE=,1,2024-09-15 16:31:07.718000+00:00,None,1280,720,29.970030,7.680590,230,"mov,mp4,m4a,3gp,3g2,mj2",h264
3,gs://datachain-kinetics-700-2020,val/abseiling/Kbmv94BqZzU_000072_000082.mp4,3315420,1726417867526305,CKGh6d2vxYgDEAE=,1,2024-09-15 16:31:07.528000+00:00,None,1280,720,29.970030,10.012811,300,"mov,mp4,m4a,3gp,3g2,mj2",h264
4,gs://datachain-kinetics-700-2020,val/abseiling/LFMlCcsLdKA_000126_000136.mp4,611088,1726417866815279,CK/uvd2vxYgDEAE=,1,2024-09-15 16:31:06.818000+00:00,None,320,240,25.000000,10.021791,251,"mov,mp4,m4a,3gp,3g2,mj2",h264
5,gs://datachain-kinetics-700-2020,val/abseiling/MpnTZgY650k_000037_000047.mp4,493020,1726417866998681,CJmHyd2vxYgDEAE=,1,2024-09-15 16:31:07+00:00,None,480,270,25.000000,10.019795,250,"mov,mp4,m4a,3gp,3g2,mj2",h264
6,gs://datachain-kinetics-700-2020,val/abseiling/ND-5jaduGFg_000024_000034.mp4,312869,1726417866659017,CMmptN2vxYgDEAE=,1,2024-09-15 16:31:06.661000+00:00,None,320,240,11.988012,10.016802,120,"mov,mp4,m4a,3gp,3g2,mj2",h264
7,gs://datachain-kinetics-700-2020,val/abseiling/NHXo1PE12BU_000075_000085.mp4,512350,1726417867016976,CJCWyt2vxYgDEAE=,1,2024-09-15 16:31:07.019000+00:00,None,320,240,29.970030,10.010000,300,"mov,mp4,m4a,3gp,3g2,mj2",h264
8,gs://datachain-kinetics-700-2020,val/abseiling/NkLeMkOcftM_000037_000047.mp4,228082,1726417866405677,CK3upN2vxYgDEAE=,1,2024-09-15 16:31:06.408000+00:00,None,266,212,25.000000,10.019795,250,"mov,mp4,m4a,3gp,3g2,mj2",h264



[Limited by 10 rows]


## Extracting Video Fragments

We are going to split each video file into 1-second fragments.

Let’s create a new generator for this and run this generator against our DataChain:

In [11]:
def split_fragments(file: VideoFile, meta: Video) -> Iterator[VideoFragment]:
    intervals = [
        (start, min(start + 1, meta.duration))
        for start in range(0, int(meta.duration))
    ]
    yield from file.save_fragments(intervals, "gs://datachain-test-vlad/kinetics/fragments")

video_fragments_dc = video_meta_dc.gen(fragment=split_fragments).save()

Cleanup:   0%|          | 0/1 [00:00<?, ? tables/s]

In [12]:
video_fragments_dc.print_schema()

 fragment: VideoFragment@v1
     source: str
     path: str
     size: int
     version: str
     etag: str
     is_latest: bool
     last_modified: datetime
     location: Union[dict, list[dict], NoneType]
     start: float
     end: float


In [13]:
video_fragments_dc.show(10)

,fragment,fragment,fragment,fragment,fragment,fragment,fragment,fragment,fragment,fragment
,source,path,size,version,etag,is_latest,last_modified,location,start,end
0,gs://datachain-test-vlad/kinetics/fragments,ND-5jaduGFg_000024_000034_000_1000.mp4,33806,1738242921930423,CLe9lL/DnYsDEAE=,1,2025-01-30 13:15:22.052000+00:00,None,0.0,1.0
1,gs://datachain-test-vlad/kinetics/fragments,ND-5jaduGFg_000024_000034_1000_2000.mp4,37076,1738242923406421,CNXI7r/DnYsDEAE=,1,2025-01-30 13:15:23.520000+00:00,None,1.0,2.0
2,gs://datachain-test-vlad/kinetics/fragments,ND-5jaduGFg_000024_000034_2000_3000.mp4,41185,1738242924876355,CMOkyMDDnYsDEAE=,1,2025-01-30 13:15:24.993000+00:00,None,2.0,3.0
3,gs://datachain-test-vlad/kinetics/fragments,ND-5jaduGFg_000024_000034_3000_4000.mp4,37247,1738242926335304,CMiqocHDnYsDEAE=,1,2025-01-30 13:15:26.444000+00:00,None,3.0,4.0
4,gs://datachain-test-vlad/kinetics/fragments,ND-5jaduGFg_000024_000034_4000_5000.mp4,37151,1738242927316691,CNOd3cHDnYsDEAE=,1,2025-01-30 13:15:27.430000+00:00,None,4.0,5.0
5,gs://datachain-test-vlad/kinetics/fragments,ND-5jaduGFg_000024_000034_5000_6000.mp4,36273,1738242928276253,CJ3ml8LDnYsDEAE=,1,2025-01-30 13:15:28.407000+00:00,None,5.0,6.0
6,gs://datachain-test-vlad/kinetics/fragments,ND-5jaduGFg_000024_000034_6000_7000.mp4,38547,1738242929312368,CPCE18LDnYsDEAE=,1,2025-01-30 13:15:29.411000+00:00,None,6.0,7.0
7,gs://datachain-test-vlad/kinetics/fragments,ND-5jaduGFg_000024_000034_7000_8000.mp4,41958,1738242930345222,CIaKlsPDnYsDEAE=,1,2025-01-30 13:15:30.455000+00:00,None,7.0,8.0
8,gs://datachain-test-vlad/kinetics/fragments,ND-5jaduGFg_000024_000034_8000_9000.mp4,32325,1738242931284626,CJK1z8PDnYsDEAE=,1,2025-01-30 13:15:31.407000+00:00,None,8.0,9.0



[Limited by 10 rows]


## Extracting Video Frames

We are going to extract every 10th frame from each video file.

Let’s create a new generator for this and run this generator against our DataChain:

In [14]:
def split_frames(file: VideoFile) -> Iterator[VideoFrame]:
    yield from file.save_frames("gs://datachain-test-vlad/kinetics/frames", step=10)

video_frames_dc = video_meta_dc.gen(frame=split_frames).save()

Cleanup:   0%|          | 0/2 [00:00<?, ? tables/s]

In [15]:
video_frames_dc.print_schema()

 frame: VideoFrame@v1
     source: str
     path: str
     size: int
     version: str
     etag: str
     is_latest: bool
     last_modified: datetime
     location: Union[dict, list[dict], NoneType]
     frame: int
     timestamp: float


In [16]:
video_frames_dc.show(10)

,frame,frame,frame,frame,frame,frame,frame,frame,frame,frame
,source,path,size,version,etag,is_latest,last_modified,location,frame,timestamp
0,gs://datachain-test-vlad/kinetics/frames,LFMlCcsLdKA_000126_000136_000000.jpg,15918,1738242980560113,CPH5jtvDnYsDEAE=,1,2025-01-30 13:16:20.668000+00:00,None,0,0.0
1,gs://datachain-test-vlad/kinetics/frames,LFMlCcsLdKA_000126_000136_000010.jpg,15942,1738242981739698,CLL51tvDnYsDEAE=,1,2025-01-30 13:16:21.848000+00:00,None,10,0.4
2,gs://datachain-test-vlad/kinetics/frames,LFMlCcsLdKA_000126_000136_000020.jpg,16428,1738242982740011,CKuAlNzDnYsDEAE=,1,2025-01-30 13:16:22.852000+00:00,None,20,0.8
3,gs://datachain-test-vlad/kinetics/frames,LFMlCcsLdKA_000126_000136_000030.jpg,16272,1738242983485966,CI7EwdzDnYsDEAE=,1,2025-01-30 13:16:23.590000+00:00,None,30,1.2
4,gs://datachain-test-vlad/kinetics/frames,LFMlCcsLdKA_000126_000136_000040.jpg,16600,1738242984371706,CPrL99zDnYsDEAE=,1,2025-01-30 13:16:24.474000+00:00,None,40,1.6
5,gs://datachain-test-vlad/kinetics/frames,LFMlCcsLdKA_000126_000136_000050.jpg,16812,1738242985152356,COSep93DnYsDEAE=,1,2025-01-30 13:16:25.272000+00:00,None,50,2.0
6,gs://datachain-test-vlad/kinetics/frames,LFMlCcsLdKA_000126_000136_000060.jpg,17018,1738242985952003,CIOG2N3DnYsDEAE=,1,2025-01-30 13:16:26.062000+00:00,None,60,2.4
7,gs://datachain-test-vlad/kinetics/frames,LFMlCcsLdKA_000126_000136_000070.jpg,16945,1738242986720284,CJz4ht7DnYsDEAE=,1,2025-01-30 13:16:26.822000+00:00,None,70,2.8
8,gs://datachain-test-vlad/kinetics/frames,LFMlCcsLdKA_000126_000136_000080.jpg,16723,1738242987635931,CNvpvt7DnYsDEAE=,1,2025-01-30 13:16:27.753000+00:00,None,80,3.2



[Limited by 10 rows]
